In [ ]:
!pip install scikit-learn
import numpy as np
from sklearn.decomposition import IncrementalPCA

# Simulated client data
client_data = [
    np.random.randn(100, 10),  # client 1
    np.random.randn(80, 10),   # client 2
    np.random.randn(120, 10)   # client 3
]

n_components = 5  # number of PCs we want

# Step 1: Each client does incremental PCA locally
client_ipca_results = []
for X in client_data:
    ipca = IncrementalPCA(n_components=n_components, batch_size=20)
    ipca.partial_fit(X)  # could also do in smaller batches if needed
    # Save local mean and components
    client_ipca_results.append({
        'mean': ipca.mean_,
        'components': ipca.components_,
        'explained_variance': ipca.explained_variance_
    })

# Step 2: Aggregate covariances on server
global_cov = np.zeros((X.shape[1], X.shape[1])) # X.shape[1] : num of features
total_samples = 0

for i, X in enumerate(client_data):
    X_centered = X - client_ipca_results[i]['mean']
    cov_local = X_centered.T @ X_centered
    global_cov += cov_local
    total_samples += X.shape[0]

global_cov /= total_samples

# Step 3: Compute global PCA from aggregated covariance
eigvals, eigvecs = np.linalg.eigh(global_cov)
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

print("Global top 3 PCs:\n", eigvecs[:, :3])

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 6.6 MB/s eta 0:00:00a 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 6.5 MB/s eta 0:00:0000:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Global top 3 PCs:
 [[ 1.39983042e-01 -3.13351086e-01 -2.29853637e-01]
 [ 3.40463685e-01  2.98973531e-01  1.07784830e-01]
 [-1.81969352e-01  4.89814110e-02 -4.79603566e-01]
 [-4.09003387e-01  5.60573378e-01 -2.86427010e-01]
 [-3.10901610e-01 -2.09413450e-01  4.44685670e-01]
 [ 9.57347204e-02 -5.12478415e-02 -2.16905292e-01]
 [-1.72505624e-01 -1.73547908e-01  4.65303263e-01]
 [-6.71045228e-01 -6.23090087e-02 -6.22087746e-02]
 [-3.46212004e-04  6.20637817e-01  3.95849281e-01]
 [-2.79656316e-01 -1.73526109e-01  4.02835225e-02]]
